In [19]:
import re
import regex
import json
import unicodedata
from dataclasses import dataclass
from typing import Dict, List, Optional, Any

import pandas as pd
from unidecode import unidecode

import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding,
    pipeline
)

In [20]:
BERTIMBAU_MODEL = 'neuralmind/bert-base-portuguese-cased'

DEVICE = 0  if torch.cuda.is_available() else -1
MAX_LENGTH = 512
RANDOM_STATE = 42

In [21]:
def normalize_whitespace(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def remove_xml_tags(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    return normalize_whitespace(text)


def normalize_text_for_model(text: str) -> str:
    """
    Versão levemente normalizada para classificação.
    Preserva o conteúdo sem achatar demais a linguagem jurídica/administrativa.
    """
    text = remove_xml_tags(text)
    text = text.replace("–", "-").replace("—", "-")
    text = normalize_whitespace(text)
    return text


def normalize_text_for_matching(text: str) -> str:
    """
    Versão mais agressiva para matching e dicionários.
    """
    text = normalize_text_for_model(text)
    text = unidecode(text.lower())
    return text


def build_model_input(row: pd.Series, body_chars: int = 1500) -> str:
    titulo = str(row.get("titulo", "") or "")
    texto = str(row.get("texto", "") or "")
    texto = normalize_text_for_model(texto)
    titulo = normalize_text_for_model(titulo)

    corpo_curto = texto[:body_chars]
    final_text = f"TITULO: {titulo} [SEP] TEXTO: {corpo_curto}"
    return final_text[:4000]

In [ ]:
VALOR_PATTERN = re.compile(
    r'R\$\s?(?:\d{1,3}(?:\.\d{3})*|\d+)(?:,\d{2})?'
)

CNPJ_PATTERN = re.compile(
    r'\b\d{2}\.?\d{3}\.?\d{3}/?\d{4}-?\d{2}\b'
)

PROCESSO_PATTERN = re.compile(
    r'\b(?:processo|proc\.)\s*(?:n[oº°]\s*)?([\d./-]+)',
    flags=re.IGNORECASE
)

CONTRATO_PATTERN = re.compile(
    r'\b(?:contrato)\s*(?:n[oº°]\s*)?([\w./-]+)',
    flags=re.IGNORECASE
)

DATA_PATTERN = re.compile(
    r'\b\d{1,2}/\d{1,2}/\d{2,4}\b|\b\d{1,2}\s+de\s+[A-Za-zçÇãõáéíóúâêô]+\s+de\s+\d{4}\b',
    flags=re.IGNORECASE
)

NORMA_PATTERN = re.compile(
    r'\b(?:lei|decreto|portaria|instrucao normativa|medida provisoria|resolucao|norma)\s*(?:n[oº°]\s*)?[\d./-]+',
    flags=re.IGNORECASE
)

In [23]:
def extract_first(pattern: re.Pattern, text: str) -> Optional[str]:
    if not isinstance(text, str):
        return None
    match = pattern.search(text)
    return match.group(1).strip() if match and match.groups() else (match.group(0).strip() if match else None)

def extract_all(pattern: re.Pattern, text:str) -> List[str]:
    if not isinstance(text, str):
        return []
    matches = pattern.findall(text)
    
    if not matches:
        return []
    
    if isinstance(matches[0], tuple):
        flat = []
        for item in matches:
            for x in item:
                if x:
                    flat.append(x.strip())
        return list(dict.fromkeys(flat))
    
    return list(dict.fromkeys([m.strip() for m in matches if isinstance(m, str) and m.strip()]))

def extract_structured_entities(text: str) -> Dict[str, Any]:
    return {
        'valor_monetario': extract_all(VALOR_PATTERN, text),
        'cnpj': extract_all(CNPJ_PATTERN, text), 
        'num_processo': extract_all(PROCESSO_PATTERN, text),
        'num_contrato': extract_all(CONTRATO_PATTERN, text),
        'datas_mencionadas': extract_all(DATA_PATTERN, text),
        'fundamentos_legais': extract_all(NORMA_PATTERN, text),
    }

In [24]:
FORCA_MAP = {
    'marinha do brasil': 'MARINHA',
    'comando da marinha': 'MARINHA', 
    'mb': 'MARINHA', 
    'exercito brasileiro': 'EXERCITO', 
    'comando do exercito': 'EXERCITO',
    'eb': 'EXERCITO', 
    'forca aerea brasileira': 'FORCA AEREA',
    'comando da aeronautica': 'FORCA AEREA',
    'fab': 'FORCA AEREA',
}

def detect_forca(text: str) -> Optional[str]:
    text_norm = normalize_text_for_matching(text)
    for key, value in FORCA_MAP.items():
        if key in text_norm:
            return value
    return None

In [25]:
LABELS = [
    'ATOS_PESSOAL',
    'LICITACAO', 
    'CONTRATO', 
    'NORMATIVO', 
    'ORCAMENTARIO_FINANCEIRO',
    'PUNITIVO_DISCIPLINAR',
    'CONVENIO',
    'HOMOLOGACAO', 
    'AUTORIZACAO_FUNCIONAMENTO',
    'OUTROS'
]

label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

In [26]:
class DocumentClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], 
            truncation=True, 
            max_length=self.max_length,
            padding=False,
            return_tensors='pt'
        )
        
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item     
        

In [27]:
def prepare_classification_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['model_input'] = df.apply(build_model_input, axis=1)
    df['label_id'] = df['label'].map(label2id)
    df = df.dropna(subset=[ 'model_input', 'label_id'])
    return df

In [28]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro')
    }

In [29]:
def train_classifier(df_train: pd.DataFrame, output_dir: str = './models/bertimbau_tipo_doc'):
    df_train = prepare_classification_dataframe(df_train)
    
    train_df, valid_df = train_test_split(
        df_train, 
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=df_train['label_id']
    )
    
    tokenizer = AutoTokenizer.from_pretrained(BERTIMBAU_MODEL)
    
    train_dataset = DocumentClassificationDataset(
        texts=train_df['model_input'].tolist(),
        labels=train_df['label_id'].tolist(),
        tokenizer=tokenizer,
        max_length=MAX_LENGTH
    )
    
    valid_dataset = DocumentClassificationDataset(
        texts=valid_df['model_input'].tolist(),
        labels=valid_df['label_id'].tolist(),
        tokenizer=tokenizer,
        max_length=MAX_LENGTH
    )
    
    model = AutoModelForSequenceClassification.from_pretrained(
        BERTIMBAU_MODEL,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id
    )
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='epoch',
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1', 
        greater_is_better=True,
        save_total_limit=2,
        report_to='none'
    )
    
    trainer = Trainer(
        model=model, 
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset, 
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    
    preds_output = trainer.predict(valid_dataset)
    preds = preds_output.predictions.argmax(axis=1)
    y_true = valid_df['label_id'].values
    
    print('\nClassification Report:\n')
    print(classification_report(y_true, preds, target_names=LABELS))
    
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    return trainer, valid_df, preds

In [30]:
def load_classifier(model_dir: str = './models/bertimbau_tipo_doc'):
    clf = pipeline(
        'text-classification',
        model=model_dir,
        tokenizer=model_dir,
        top_k=None, 
        device=DEVICE
    )
    
    return clf

In [31]:
def predict_document_type(df: pd.DataFrame, clf_pipeline, text_col: str = 'model_input') -> pd.DataFrame:
    df = df.copy()
    if text_col not in df.columns:
        df[text_col] = df.apply(build_model_input, axis=1)
        
    predictions = clf_pipeline(
        df[text_col].tolist(), 
        truncation=True,
        max_length=MAX_LENGTH
    )
    
    pred_labels = []
    pred_scores = []
    
    for pred in predictions:
        best = sorted(pred, key=lambda x: x['score'], reverse=True)[0]
        pred_labels.append(best['label'])
        pred_scores.append(best['score'])
        
    df['tipo_doc_pred'] = pred_labels
    df['tipo_doc_score'] = pred_scores
    
    return df
        

In [32]:
def enrich_documents(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    df['titulo'] = df['titulo'].fillna('')
    df['texto'] = df['texto'].fillna('')
    
    df['texto_limpo'] = df['texto'].apply(normalize_text_for_model)
    df['model_limpo'] = df.apply(build_model_input, axis=1)
    
    extracted = df['texto_limpo'].apply(extract_structured_entities)
    extracted_df = pd.json_normalize(extracted)
    
    df = pd.concat([df, extracted_df], axis=1)
    
    df['forca_detectada'] = (
        df['titulo'].fillna('') + ' ' + df['texto_limpo'].fillna('')
    ).apply(detect_forca)
    
    return df

In [33]:
def run_pipeline(
    df: pd.DataFrame, 
    classifier_pipeline=None
) -> pd.DataFrame:
    df = enrich_documents(df)
    
    if classifier_pipeline is not None:
        df = predict_document_type(df, classifier_pipeline)
    
    return df

In [34]:
import polars as pl

textos = pl.read_parquet('../database/textos/*.parquet').select('id', 'texto')
metadata = pl.read_parquet('../database/metadata/*.parquet').select('id', 'pubName', 'name', 'artType', 'pubDate', 'artCategory', 'pdfPage')
df = textos.join(metadata, on='id', how='inner').to_pandas()
del textos, metadata

df.head()

,id,texto,pubName,name,artType,pubDate,artCategory,pdfPage
0,12002010442,"PORTARIA DAC Nº 1.747/SIE, DE 26 DE DEZEMBRO D...",DO1,PORTARIA,PORTARIA,04/01/2002,"Ministério da Agricultura,/Pecuária e Abasteci...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...
1,12002010443,"PORTARIA DAC Nº 1.748/SIE, DE 26 DE DEZEMBRO D...",DO1,PORTARIA,PORTARIA,04/01/2002,"Ministério da Agricultura,/Pecuária e Abasteci...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...
2,12002010444,"PORTARIA DAC Nº 1.749/SIE, DE 26 DE DEZEMBRO D...",DO1,PORTARIA,PORTARIA,04/01/2002,"Ministério da Agricultura,/Pecuária e Abasteci...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...
3,12002010445,"PORTARIA Nº 45/DAdM, DE 27 DE DEZEMBRO DE 2001...",DO1,PORTARIA,PORTARIA,04/01/2002,"Ministério da Agricultura,/Pecuária e Abasteci...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...
4,12002010746,"PORTARIA INTERMINISTERIAL Nº 2, DE 3 DE JANEIR...",DO1,PORTARIA,PORTARIA,07/01/2002,Ministério da Defesa/GABINETE DO MINISTRO,http://pesquisa.in.gov.br/imprensa/jsp/visuali...


In [35]:
df.rename(columns={'id':'doc_id', 'pubDate':'data_pub', 'name':'titulo'}, inplace=True)

In [36]:
df_out = run_pipeline(df, classifier_pipeline=None)

df_out

,doc_id,texto,pubName,titulo,artType,data_pub,artCategory,pdfPage
0,12002010442,"PORTARIA DAC Nº 1.747/SIE, DE 26 DE DEZEMBRO D...",DO1,PORTARIA,PORTARIA,04/01/2002,"Ministério da Agricultura,/Pecuária e Abasteci...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...
1,12002010443,"PORTARIA DAC Nº 1.748/SIE, DE 26 DE DEZEMBRO D...",DO1,PORTARIA,PORTARIA,04/01/2002,"Ministério da Agricultura,/Pecuária e Abasteci...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...
2,12002010444,"PORTARIA DAC Nº 1.749/SIE, DE 26 DE DEZEMBRO D...",DO1,PORTARIA,PORTARIA,04/01/2002,"Ministério da Agricultura,/Pecuária e Abasteci...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...
3,12002010445,"PORTARIA Nº 45/DAdM, DE 27 DE DEZEMBRO DE 2001...",DO1,PORTARIA,PORTARIA,04/01/2002,"Ministério da Agricultura,/Pecuária e Abasteci...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...
4,12002010746,"PORTARIA INTERMINISTERIAL Nº 2, DE 3 DE JANEIR...",DO1,PORTARIA,PORTARIA,07/01/2002,Ministério da Defesa/GABINETE DO MINISTRO,http://pesquisa.in.gov.br/imprensa/jsp/visuali...
...,...,...,...,...,...,...,...,...
1262525,530_20251231_23469728,EXTRATO DE CREDENCIAMENTO Nº 38/2025 - UASG 78...,DO3,2025-12-30/11503645/COMPRASNETWS,Extrato de Credenciamento,31/12/2025,Ministério da Defesa/Comando da Marinha/Comand...,http://pesquisa.in.gov.br/imprensa/jsp/visuali...
1262526,530_20251231_23470217,EXTRATO DE CREDENCIAMENTO Nº 37/2025 - UASG 78...,DO3,2025-12-30/11503746/COMPRASNETWS,Extrato de Credenciamento,31/12/2025,Ministério da Defesa/Comando da Marinha/Comand...,http://pesquisa.in.gov.br/imprensa/jsp/visuali...
1262527,530_20251231_23470438,EXTRATO DE CREDENCIAMENTO Nº 20/2025 - UASG 78...,DO3,2025-12-30/11503765/COMPRASNETWS,Extrato de Credenciamento,31/12/2025,Ministério da Defesa/Comando da Marinha/Comand...,http://pesquisa.in.gov.br/imprensa/jsp/visuali...
1262528,530_20251231_23470552,EXTRATO DE CREDENCIAMENTO Nº 31/2025 - UASG 78...,DO3,2025-12-30/11503761/COMPRASNETWS,Extrato de Credenciamento,31/12/2025,Ministério da Defesa/Comando da Marinha/Comand...,http://pesquisa.in.gov.br/imprensa/jsp/visuali...


In [37]:
df_out

,doc_id,texto,pubName,titulo,artType,data_pub,artCategory,pdfPage,texto_limpo,model_limpo,valor_monetario,cnpj,num_processo,num_contrato,datas_mencionadas,fundamentos_legais,forca_detectada
0,12002010442,"PORTARIA DAC Nº 1.747/SIE, DE 26 DE DEZEMBRO D...",DO1,PORTARIA,PORTARIA,04/01/2002,"Ministério da Agricultura,/Pecuária e Abasteci...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,"PORTARIA DAC Nº 1.747/SIE, DE 26 DE DEZEMBRO D...",TITULO: PORTARIA [SEP] TEXTO: PORTARIA DAC Nº ...,[],[],[],[ole],"[26 DE DEZEMBRO DE 2001, 21 de dezembro de 200...","[Portaria nº 467/, Lei nº 7565, Portaria nº 533/]",MARINHA
1,12002010443,"PORTARIA DAC Nº 1.748/SIE, DE 26 DE DEZEMBRO D...",DO1,PORTARIA,PORTARIA,04/01/2002,"Ministério da Agricultura,/Pecuária e Abasteci...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,"PORTARIA DAC Nº 1.748/SIE, DE 26 DE DEZEMBRO D...",TITULO: PORTARIA [SEP] TEXTO: PORTARIA DAC Nº ...,[],[],[],[ole],"[26 DE DEZEMBRO DE 2001, 21 de dezembro de 200...","[Portaria nº 467/, Lei nº 7565, Portaria nº 533/]",MARINHA
2,12002010444,"PORTARIA DAC Nº 1.749/SIE, DE 26 DE DEZEMBRO D...",DO1,PORTARIA,PORTARIA,04/01/2002,"Ministério da Agricultura,/Pecuária e Abasteci...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,"PORTARIA DAC Nº 1.749/SIE, DE 26 DE DEZEMBRO D...",TITULO: PORTARIA [SEP] TEXTO: PORTARIA DAC Nº ...,[],[],[],[ole],"[26 DE DEZEMBRO DE 2001, 21 de dezembro de 200...","[Portaria nº 467/, Lei nº 7565, Portaria nº 533/]",MARINHA
3,12002010445,"PORTARIA Nº 45/DAdM, DE 27 DE DEZEMBRO DE 2001...",DO1,PORTARIA,PORTARIA,04/01/2002,"Ministério da Agricultura,/Pecuária e Abasteci...",http://pesquisa.in.gov.br/imprensa/jsp/visuali...,"PORTARIA Nº 45/DAdM, DE 27 DE DEZEMBRO DE 2001...",TITULO: PORTARIA [SEP] TEXTO: PORTARIA Nº 45/D...,[],[],[],[],"[27 DE DEZEMBRO DE 2001, 25 de julho de 2000, ...","[PORTARIA Nº 45/, Lei nº 9.995, Portaria nº 10...",MARINHA
4,12002010746,"PORTARIA INTERMINISTERIAL Nº 2, DE 3 DE JANEIR...",DO1,PORTARIA,PORTARIA,07/01/2002,Ministério da Defesa/GABINETE DO MINISTRO,http://pesquisa.in.gov.br/imprensa/jsp/visuali...,"PORTARIA INTERMINISTERIAL Nº 2, DE 3 DE JANEIR...",TITULO: PORTARIA [SEP] TEXTO: PORTARIA INTERMI...,[],[],[],[],"[3 DE JANEIRO DE 2002, 11 de fevereiro de 2000...","[Decreto nº 3.363, Portaria.]",MARINHA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1262525,530_20251231_23469728,EXTRATO DE CREDENCIAMENTO Nº 38/2025 - UASG 78...,DO3,2025-12-30/11503645/COMPRASNETWS,Extrato de Credenciamento,31/12/2025,Ministério da Defesa/Comando da Marinha/Comand...,http://pesquisa.in.gov.br/imprensa/jsp/visuali...,EXTRATO DE CREDENCIAMENTO Nº 38/2025 - UASG 78...,TITULO: 2025-12-30/11503645/COMPRASNETWS [SEP]...,"[R$ 151.062,00]",[08.242.471/0001-18],[],"[atante, atado]","[15/12/2025, 14/12/2026, 30/12/2025]",[],NaN
1262526,530_20251231_23470217,EXTRATO DE CREDENCIAMENTO Nº 37/2025 - UASG 78...,DO3,2025-12-30/11503746/COMPRASNETWS,Extrato de Credenciamento,31/12/2025,Ministério da Defesa/Comando da Marinha/Comand...,http://pesquisa.in.gov.br/imprensa/jsp/visuali...,EXTRATO DE CREDENCIAMENTO Nº 37/2025 - UASG 78...,TITULO: 2025-12-30/11503746/COMPRASNETWS [SEP]...,"[R$ 20.000,00]",[60.143.972/0001-67],[],"[atante, atado]","[15/12/2025, 14/12/2026, 30/12/2025]",[],NaN
1262527,530_20251231_23470438,EXTRATO DE CREDENCIAMENTO Nº 20/2025 - UASG 78...,DO3,2025-12-30/11503765/COMPRASNETWS,Extrato de Credenciamento,31/12/2025,Ministério da Defesa/Comando da Marinha/Comand...,http://pesquisa.in.gov.br/imprensa/jsp/visuali...,EXTRATO DE CREDENCIAMENTO Nº 20/2025 - UASG 78...,TITULO: 2025-12-30/11503765/COMPRASNETWS [SEP]...,"[R$ 20.000,00]",[70.030.606/0001-55],[],"[atante, atado]","[09/12/2025, 08/12/2026, 30/12/2025]",[],NaN
1262528,530_20251231_23470552,EXTRATO DE CREDENCIAMENTO Nº 31/2025 - UASG 78...,DO3,2025-12-30/11503761/COMPRASNETWS,Extrato de Credenciamento,31/12/2025,Ministério da Defesa/Comando da Marinha/Comand...,http://pesquisa.in.gov.br/imprensa/jsp/vis

In [42]:
df_out[df_out.num_processo!=list()]

ValueError: ('Lengths must match to compare', (1262530,), (0,))